# benchmarking_helpers.ipynb
**Helper Functions for Cell Type Annotation Benchmarking (Python Version)**

Contains utility functions for metrics, cross-validation, and result aggregation.
This is a Python/scanpy translation of the R benchmarking_helpers.R file.

In [ ]:
# Import required libraries
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import time
import warnings
from typing import Dict, List, Tuple, Any, Optional
import scipy.sparse as sp
from collections import defaultdict

## Performance Metrics Calculation

In [ ]:
def calculate_metrics(predicted: List[str], true_labels: List[str]) -> Dict[str, Any]:
    """
    Calculate Comprehensive Performance Metrics
    
    Purpose: Computes accuracy, precision, recall, F1, macro F1, and Cohen's Kappa
    Inputs: 
      - predicted: List of predicted cell types
      - true_labels: List of ground truth cell types  
    Outputs: Dict with overall_accuracy, macro_f1, cohens_kappa, per_class_metrics dataframe
    Key Logic: Handles per-class TP/FP/FN calculations, excludes "Unknown" predictions
    """
    
    # Convert to numpy arrays
    predicted = np.array(predicted)
    true_labels = np.array(true_labels)
    
    # Overall Accuracy
    overall_accuracy = accuracy_score(true_labels, predicted)
    
    # Get unique cell types (exclude Unknown and NaN)
    cell_types = np.unique(np.concatenate([predicted, true_labels]))
    cell_types = cell_types[~pd.isnull(cell_types)]
    cell_types = cell_types[cell_types != "Unknown"]
    
    # Initialize metric arrays
    precision_scores = np.zeros(len(cell_types))
    recall_scores = np.zeros(len(cell_types))
    f1_scores = np.zeros(len(cell_types))
    
    # Calculate per-class metrics
    for i, ct in enumerate(cell_types):
        # TP, FP, FN calculations
        TP = np.sum((predicted == ct) & (true_labels == ct))
        FP = np.sum((predicted == ct) & (true_labels != ct))
        FN = np.sum((predicted != ct) & (true_labels == ct))
        
        # Precision and Recall
        precision_scores[i] = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall_scores[i] = TP / (TP + FN) if (TP + FN) > 0 else 0
        
        # F1 Score
        if precision_scores[i] + recall_scores[i] > 0:
            f1_scores[i] = 2 * (precision_scores[i] * recall_scores[i]) / (precision_scores[i] + recall_scores[i])
        else:
            f1_scores[i] = 0
    
    # Macro F1 Score (unweighted average)
    macro_f1 = np.mean(f1_scores)
    
    # Cohen's Kappa
    # p_o = observed agreement (overall accuracy, includes Unknown-predicted cells)
    # p_e = expected agreement by chance, summed over filtered cell_types only
    # Unknown predictions reduce p_o and dilute marginal rates in p_e implicitly
    n = len(predicted)
    p_o = overall_accuracy
    p_e = sum(
        (np.sum(predicted == ct) / n) * (np.sum(true_labels == ct) / n)
        for ct in cell_types
    )
    cohens_kappa = (p_o - p_e) / (1 - p_e) if (1 - p_e) > 0 else 1.0
    
    # Create per-class metrics dataframe
    per_class_metrics = pd.DataFrame({
        'cell_type': cell_types,
        'precision': precision_scores,
        'recall': recall_scores,
        'f1': f1_scores
    })
    
    return {
        'overall_accuracy': overall_accuracy,
        'macro_f1': macro_f1,
        'cohens_kappa': cohens_kappa,
        'per_class_metrics': per_class_metrics
    }

## Cross-Validation Fold Creation

In [ ]:
def create_cv_folds(adata: ad.AnnData, k: int = 5, group_by: Optional[str] = None, 
                   stratify_by: str = "Ground_Truth_Celltype", seed: int = 123) -> List[List[str]]:
    """
    Create K-Fold Cross-Validation Splits
    
    Purpose: Creates k folds either grouped by donor or stratified by cell type
    Inputs:
      - adata: AnnData object with metadata
      - k: Number of folds (default 5)
      - group_by: Column for grouped CV (e.g. "donor_id"), None for stratified
      - stratify_by: Column for stratified CV (e.g. "Ground_Truth_Celltype")
      - seed: Random seed for reproducibility
    Outputs: List of k lists, each containing cell IDs for test set of that fold
    Key Logic: Grouped mode keeps donor samples together, stratified mode balances cell types
    """
    
    np.random.seed(seed)
    
    fold_cell_lists = []
    
    if group_by is not None and group_by in adata.obs.columns:
        # --- GROUPED MODE ---
        print(f"Mode: Grouped K-Fold CV by '{group_by}'.")
        
        all_groups = adata.obs[group_by].unique()
        if len(all_groups) < k:
            raise ValueError("Number of groups is less than k.")
        
        # Shuffle groups and assign to folds
        shuffled_groups = np.random.permutation(all_groups)
        group_folds = np.array_split(shuffled_groups, k)
        
        for i in range(k):
            test_groups = group_folds[i]
            # Get cell barcodes belonging to the test groups for this fold
            test_mask = adata.obs[group_by].isin(test_groups)
            fold_cell_lists.append(adata.obs.index[test_mask].tolist())
    
    else:
        # --- STRATIFIED MODE ---
        print(f"Mode: Stratified K-Fold CV by '{stratify_by}'.")
        
        if stratify_by not in adata.obs.columns:
            raise ValueError("Stratification variable not found in metadata.")
        
        # Use scikit-learn StratifiedKFold
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
        
        # Get stratification labels
        stratify_labels = adata.obs[stratify_by]
        
        # Create folds
        for train_idx, test_idx in skf.split(np.arange(len(adata.obs)), stratify_labels):
            fold_cell_lists.append(adata.obs.index[test_idx].tolist())
    
    print("Fold creation complete.")
    
    # Print fold statistics
    for i, fold_cells in enumerate(fold_cell_lists):
        print(f"Fold {i+1}: {len(fold_cells)} cells")
    
    return fold_cell_lists

## Fold Data Processing with Caching

In [ ]:
# Cache system for fold data
fold_cache = {}

import scanpy as sc
import pandas as pd
from typing import List, Dict, Any

def get_fold_data_cached(adata: sc.AnnData, folds: List[List[str]], fold_index: int, label_key: str = "Ground_Truth_Celltype") -> Dict[str, Any]:
    """
    Get Fold Data with Caching (Scanpy version)
    
    Purpose:
        Splits data into train/test for a fold and finds marker genes with caching.
    
    Inputs:
        - adata: Full AnnData object
        - folds: List of folds (each fold is a list of cell names)
        - fold_index: Which fold to process (0-based index)
        - label_key: Column in adata.obs indicating true cell type labels
    
    Outputs:
        Dict with keys: 'train', 'test', 'markers', 'fold_info'
    """
    cache_key = f"fold_{fold_index}"
    
    if cache_key not in fold_cache:
        print(f"    Computing fold {fold_index} data and markers...")
        
        # Split data
        test_cells = folds[fold_index]
        train_cells = [c for c in adata.obs_names if c not in test_cells]
        
        adata_train = adata[train_cells].copy()
        adata_test = adata[test_cells].copy()
        
        print(f"    Training: {adata_train.n_obs} cells, Testing: {adata_test.n_obs} cells")
        
        # Check if data is already log-normalized
        # Multiple indicators to determine normalization status:
        def is_log_normalized(adata_obj):
            """
            Check if AnnData is already log-normalized
            Returns: (is_normalized, reason)
            """
            # Check 1: Look for normalization flags in .uns
            if 'log1p' in adata_obj.uns:
                return True, "log1p flag found in .uns"
            
            # Check 2: Examine data range
            # Log-normalized data typically has max values < 15
            # Raw counts typically have max values > 100
            max_val = adata_obj.X.max()
            if max_val < 20:
                return True, f"max value ({max_val:.2f}) suggests log-normalized data"
            
            # Check 3: Check for integer values (raw counts are integers)
            # Sample a subset to check
            if sp.issparse(adata_obj.X):
                sample_data = adata_obj.X.data[:min(10000, len(adata_obj.X.data))]
            else:
                sample_data = adata_obj.X.ravel()[:min(10000, adata_obj.X.size)]
            
            # If more than 90% of values are integers, likely raw counts
            int_fraction = np.mean(sample_data == np.floor(sample_data))
            if int_fraction < 0.1:
                return True, f"only {int_fraction*100:.1f}% integers suggests log-normalized"
            
            # Check 4: If max > 100, likely raw counts
            if max_val > 100:
                return False, f"max value ({max_val:.2f}) suggests raw counts"
            
            # Default: assume not normalized if uncertain
            return False, "uncertain - defaulting to raw counts assumption"
        
        is_normalized, reason = is_log_normalized(adata_train)
        print(f"    Data normalization check: {reason}")
        
        if not is_normalized:
            print("    Log-normalizing training data for marker detection...")
            # Normalize and log-transform (in-place on the copy)
            sc.pp.normalize_total(adata_train, target_sum=1e4)
            sc.pp.log1p(adata_train)
            
            print("    Log-normalizing test data to match training...")
            # Apply same normalization to test data for consistency
            sc.pp.normalize_total(adata_test, target_sum=1e4)
            sc.pp.log1p(adata_test)
        else:
            print("    Data already normalized - skipping normalization step")
        
        # Find markers (expensive - do once per fold)
        sc.tl.rank_genes_groups(
            adata_train, 
            groupby=label_key, 
            method='t-test',   # or 'wilcoxon' / 'logreg' as desired
            pts=True
        )
        
        # Convert results to a DataFrame similar to Seurat output
        groups = adata_train.obs[label_key].unique()
        markers_list = []
        for group in groups:
            scores = adata_train.uns['rank_genes_groups']['scores'][group]
            pvals = adata_train.uns['rank_genes_groups']['pvals'][group]
            logfc = adata_train.uns['rank_genes_groups']['logfoldchanges'][group]
            genes = adata_train.uns['rank_genes_groups']['names'][group]
            df = pd.DataFrame({
                'gene': genes,
                'cluster': group,
                'avg_logFC': logfc,
                'p_val': pvals,
                'score': scores
            })
            markers_list.append(df)
        
        markers = pd.concat(markers_list, ignore_index=True)
        
        print(f"Found {markers.shape[0]} marker genes across {len(groups)} cell types")
        
        # Cache results
        fold_cache[cache_key] = {
            'train': adata_train,
            'test': adata_test,
            'markers': markers,
            'fold_info': {
                'train_cells': adata_train.n_obs,
                'test_cells': adata_test.n_obs,
                'train_cell_types': adata_train.obs[label_key].value_counts(),
                'test_cell_types': adata_test.obs[label_key].value_counts()
            }
        }
    else:
        print(f"    Using cached fold {fold_index} data")
    
    return fold_cache[cache_key]


## Tool Result Aggregation

In [ ]:
def aggregate_tool_results(all_predictions: List[str], all_true_labels: List[str], 
                          all_confidence_scores: List[float], all_cell_ids: List[str],
                          fold_metrics: List[Dict], fold_runtimes: List[float],
                          fold_peak_memories: List[Optional[float]],  # NEW: Memory parameter
                          tool_name: str) -> Dict[str, Any]:
    """
    Aggregate Results from Single Tool Across All Folds
    
    Purpose: Combines CV results for one tool into summary statistics
    Inputs:
      - all_predictions: Combined predictions from all folds
      - all_true_labels: Combined true labels from all folds
      - all_confidence_scores: Combined confidence scores
      - all_cell_ids: Combined cell identifiers
      - fold_metrics: List of per-fold metric results
      - fold_runtimes: List of per-fold runtimes
      - fold_peak_memories: List of per-fold peak memory (MB)
      - tool_name: Name of the tool
    Outputs: Dict with pooled_metrics, fold_variation, runtime_stats, memory_stats, detailed_results
    Key Logic: Provides both pooled analysis (Method B) and fold variation stats (Method A)
    """
    
    # Remove failed folds (None values)
    valid_fold_metrics = [fm for fm in fold_metrics if fm is not None]
    valid_runtimes = [rt for rt in fold_runtimes if not pd.isnull(rt)]
    valid_memories = [mem for mem in fold_peak_memories if mem is not None and not pd.isnull(mem)]
    
    # Pooled metrics (Method B - preferred)
    pooled_metrics = calculate_metrics(all_predictions, all_true_labels)
    pooled_metrics['confusion_matrix'] = confusion_matrix(all_true_labels, all_predictions)
    
    # Fold-wise variation (Method A - for stability assessment)
    if len(valid_fold_metrics) > 0:
        fold_accuracies = [fm['overall_accuracy'] for fm in valid_fold_metrics]
        fold_f1s = [fm['macro_f1'] for fm in valid_fold_metrics]
        fold_kappas = [fm['cohens_kappa'] for fm in valid_fold_metrics]
        
        fold_variation = {
            'accuracy_mean': np.mean(fold_accuracies),
            'accuracy_std': np.std(fold_accuracies, ddof=1) if len(fold_accuracies) > 1 else 0,
            'accuracy_folds': fold_accuracies,
            'accuracy_ci': np.mean(fold_accuracies) + np.array([-1.96, 1.96]) * (np.std(fold_accuracies, ddof=1) / np.sqrt(len(fold_accuracies))) if len(fold_accuracies) > 1 else [np.mean(fold_accuracies), np.mean(fold_accuracies)],
            
            'f1_mean': np.mean(fold_f1s),
            'f1_std': np.std(fold_f1s, ddof=1) if len(fold_f1s) > 1 else 0,
            'f1_folds': fold_f1s,
            'f1_ci': np.mean(fold_f1s) + np.array([-1.96, 1.96]) * (np.std(fold_f1s, ddof=1) / np.sqrt(len(fold_f1s))) if len(fold_f1s) > 1 else [np.mean(fold_f1s), np.mean(fold_f1s)],

            'kappa_mean': np.mean(fold_kappas),
            'kappa_std': np.std(fold_kappas, ddof=1) if len(fold_kappas) > 1 else 0,
            'kappa_folds': fold_kappas,
            'kappa_ci': np.mean(fold_kappas) + np.array([-1.96, 1.96]) * (np.std(fold_kappas, ddof=1) / np.sqrt(len(fold_kappas))) if len(fold_kappas) > 1 else [np.mean(fold_kappas), np.mean(fold_kappas)]
        }
    else:
        fold_variation = {
            'accuracy_mean': np.nan, 'accuracy_std': np.nan, 'accuracy_folds': [],
            'f1_mean': np.nan, 'f1_std': np.nan, 'f1_folds': [],
            'kappa_mean': np.nan, 'kappa_std': np.nan, 'kappa_folds': []
        }
    
    # Runtime statistics
    runtime_stats = {
        'mean_seconds': np.mean(valid_runtimes) if valid_runtimes else np.nan,
        'std_seconds': np.std(valid_runtimes, ddof=1) if len(valid_runtimes) > 1 else 0,
        'total_seconds': np.sum(valid_runtimes) if valid_runtimes else np.nan,
        'fold_runtimes': valid_runtimes
    }
    
    # Memory statistics
    memory_stats = {
        'mean_mb': np.mean(valid_memories) if valid_memories else np.nan,
        'std_mb': np.std(valid_memories, ddof=1) if len(valid_memories) > 1 else 0,
        'max_mb': np.max(valid_memories) if valid_memories else np.nan,
        'min_mb': np.min(valid_memories) if valid_memories else np.nan,
        'fold_memories': valid_memories
    }
    
    return {
        'tool_name': tool_name,
        'pooled_metrics': pooled_metrics,
        'fold_variation': fold_variation,
        'runtime_stats': runtime_stats,
        'memory_stats': memory_stats,
        'detailed_results': {
            'predictions': all_predictions,
            'true_labels': all_true_labels,
            'confidence_scores': all_confidence_scores,
            'cell_ids': all_cell_ids
        },
        'num_successful_folds': len(valid_fold_metrics),
        'num_total_folds': len(fold_metrics)
    }

## Single Tool Cross-Validation Execution

In [ ]:
def run_single_tool_cv(adata: ad.AnnData, tool_function, folds: List[List[str]], tool_name: str) -> Dict[str, Any]:
    """
    Run Cross-Validation for Single Tool
    
    Purpose: Executes one tool across all CV folds with error handling
    Inputs:
      - adata: Full AnnData object
      - tool_function: Function that takes (train, test, markers) and returns predictions
      - folds: Output from create_cv_folds()
      - tool_name: Name for logging/identification
    Outputs: Aggregated results from aggregate_tool_results()
    Key Logic: Runs tool on each fold, accumulates results, handles failures gracefully
    """
    
    print(f"\n=== Running {tool_name} ===")
    
    # Storage for this tool
    all_predictions = []
    all_true_labels = []
    all_confidence_scores = []
    all_cell_ids = []
    fold_metrics = []
    fold_runtimes = []
    fold_peak_memories = []  # NEW: Track peak memory per fold
    
    # Process each fold
    for i in range(len(folds)):
        print(f"  Fold {i+1}/{len(folds)}...")
        
        # Get pre-computed fold data
        fold_data = get_fold_data_cached(adata, folds, i)
        
        # Run tool on this fold
        start_time = time.time()
        try:
            fold_result = tool_function(fold_data['train'], fold_data['test'], fold_data['markers'])
            runtime = time.time() - start_time
            
            # Validate tool output
            required_keys = ['predictions', 'true_labels', 'confidence_scores', 'cell_ids']
            if not all(key in fold_result for key in required_keys):
                raise ValueError(f"Tool function must return dict with: {required_keys}")
            
            # Accumulate results for pooled analysis
            all_predictions.extend(fold_result['predictions'])
            all_true_labels.extend(fold_result['true_labels'])
            all_confidence_scores.extend(fold_result['confidence_scores'])
            all_cell_ids.extend(fold_result['cell_ids'])
            
            # Store fold-wise metrics
            fold_metrics.append(calculate_metrics(fold_result['predictions'], fold_result['true_labels']))
            fold_runtimes.append(runtime)
            
            # NEW: Collect peak memory if available
            peak_memory_mb = fold_result.get('peak_memory_mb', None)
            fold_peak_memories.append(peak_memory_mb)
            
            # Enhanced progress message with memory
            memory_str = f", peak memory: {peak_memory_mb:.2f} MB" if peak_memory_mb is not None else ""
            print(f"    Completed in {runtime:.2f} seconds, accuracy: {fold_metrics[-1]['overall_accuracy']*100:.2f}%{memory_str}")
            
        except Exception as e:
            warnings.warn(f"Tool {tool_name} failed on fold {i+1}: {str(e)}")
            fold_metrics.append(None)
            fold_runtimes.append(np.nan)
            fold_peak_memories.append(None)  # NEW: Track None for failed folds
            print(f"    FAILED: {str(e)}")
    
    # Aggregate results for this tool (NEW: pass fold_peak_memories)
    return aggregate_tool_results(all_predictions, all_true_labels, 
                                 all_confidence_scores, all_cell_ids,
                                 fold_metrics, fold_runtimes, fold_peak_memories,
                                 tool_name)

## All Tools Comparison and Aggregation

In [ ]:
def aggregate_all_tools(all_tool_results: Dict[str, Dict]) -> Dict[str, Any]:
    """
    Aggregate Results from All Tools 
    
    Purpose: Creates comparison table and summary statistics across all tools
    Inputs:
      - all_tool_results: Dict of results from run_single_tool_cv() for each tool
    Outputs: Dict with comparison_table, detailed_results, best_tool, summary_stats
    Key Logic: Extracts key metrics from each tool, sorts by accuracy, provides rankings
    """
    
    # Create comparison table
    comparison_data = []
    
    # Extract metrics for each tool
    for tool_name, result in all_tool_results.items():
        comparison_data.append({
            'tool': tool_name,
            'pooled_accuracy': result['pooled_metrics']['overall_accuracy'],
            'pooled_macro_f1': result['pooled_metrics']['macro_f1'],
            'pooled_kappa': result['pooled_metrics']['cohens_kappa'],
            'accuracy_mean': result['fold_variation']['accuracy_mean'],
            'accuracy_std': result['fold_variation']['accuracy_std'],
            'accuracy_ci_lower': result['fold_variation']['accuracy_ci'][0],
            'accuracy_ci_upper': result['fold_variation']['accuracy_ci'][1],
            'f1_mean': result['fold_variation']['f1_mean'],
            'f1_std': result['fold_variation']['f1_std'],
            'f1_ci_lower': result['fold_variation']['f1_ci'][0],
            'f1_ci_upper': result['fold_variation']['f1_ci'][1],
            'kappa_mean': result['fold_variation']['kappa_mean'],
            'kappa_std': result['fold_variation']['kappa_std'],
            'kappa_ci_lower': result['fold_variation']['kappa_ci'][0],
            'kappa_ci_upper': result['fold_variation']['kappa_ci'][1],
            'runtime_mean': result['runtime_stats']['mean_seconds'],
            'runtime_std': result['runtime_stats']['std_seconds'],
            'peak_memory_mean_mb': result['memory_stats']['mean_mb'],
            'peak_memory_std_mb': result['memory_stats']['std_mb'],
            'peak_memory_max_mb': result['memory_stats']['max_mb'],
            'successful_folds': result['num_successful_folds']
        })
    
    # Create DataFrame and sort by pooled accuracy
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('pooled_accuracy', ascending=False).reset_index(drop=True)
    
    return {
        'comparison_table': comparison_df,
        'detailed_results': all_tool_results,
        'best_tool': comparison_df.iloc[0]['tool'] if len(comparison_df) > 0 else None,
        'summary_stats': {
            'num_tools': len(comparison_df),
            'best_accuracy': comparison_df['pooled_accuracy'].max() if len(comparison_df) > 0 else np.nan,
            'best_f1': comparison_df['pooled_macro_f1'].max() if len(comparison_df) > 0 else np.nan,
            'best_kappa': comparison_df['pooled_kappa'].max() if len(comparison_df) > 0 else np.nan,
            'fastest_tool': comparison_df.loc[comparison_df['runtime_mean'].idxmin(), 'tool'] if len(comparison_df) > 0 else None,
            'most_memory_efficient': comparison_df.loc[comparison_df['peak_memory_mean_mb'].idxmin(), 'tool'] if len(comparison_df) > 0 and not comparison_df['peak_memory_mean_mb'].isna().all() else None
        }
    }

## Initialization Message

In [ ]:
print("Benchmarking helper functions loaded successfully!")